# 15 · Policy simulation

Decision-support scenarios for VUP coverage, targeting budgets, NSDS benefit levels and district protection gaps.

| | |
|---|---|
| **Inputs** | Precomputed tables only: `data/processed/household_master.parquet`, `household_cpi_shock.parquet`, `household_scores.parquet` (out-of-sample model scores from `src/models/score_households.py`). No raw `.dta` file is read. |
| **Code** | `src/simulation/policy_simulator.py`; tests in `tests/test_policy_simulator.py` |
| **Outputs** | `data/processed/scenario_results.parquet` (aggregated scenario grid, read by the dashboard), `reports/figures/policy_simulation/*.png`, `reports/policy_simulation_summary.md` |
| **Method** | `docs/policy_simulation_methodology.md` |

**Read every number as a decision-support estimate:**
- static, first-round accounting on EICV7 2023/24 households as surveyed, using survey weights (`weight` for households, `pop_wt` for people);
- no behavioural response;
- **no causal poverty reduction is claimed**;
- model scores rank households for prioritisation and verification, and **must not be used to select or exclude individual beneficiaries**.

Transfer values use the NSDS programme rate of **30,500 RWF per household per quarter** (instruction manual; the survey median is 30,000 RWF), which can be changed.

In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "simulation" / "policy_simulator.py").exists())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import Image, display

from src.models import plots as pl
from src.simulation import policy_simulator as ps

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)
FIG = ROOT / "reports" / "figures" / "policy_simulation"
FIG.mkdir(parents=True, exist_ok=True)

df = ps.load_simulation_frame()
w, pw = df["weight"], df["pop_wt"]
base = pd.Series({
    "households (weighted)": w.sum(), "people (weighted)": pw.sum(),
    "poverty rate, % of people": 100 * np.average(df["poor"], weights=pw),
    "VUP coverage (any component), % of households": 100 * np.average(df["vup_any"], weights=w),
    "Direct Support coverage, % of households": 100 * np.average(df["vup_direct_support"], weights=w),
    "NSDS coverage, % of households": 100 * np.average(df["vup_nsds"], weights=w),
    "high-need households (weighted)": (w * df["high_need"]).sum(),
    "high-need households without VUP (weighted)": (w * df["high_need"] * (1 - df["vup_any"])).sum(),
})
base.to_frame("baseline (EICV7 2023/24)")

## 1. Coverage expansion: VUP Direct Support

How it works:
- Add *X* percentage points of all households to Direct Support.
- New beneficiaries are taken from households with **no VUP component**, in priority order: poverty-model score first, then distress score.
- Each new household receives the assumed quarterly payment.

In [ ]:
examples = {x: ps.coverage_expansion(df, x) for x in (1, 2, 5)}
cov = pd.DataFrame({f"+{x} pp": r["national"] for x, r in examples.items()}).drop(["pool"])
show = ["households_added", "people_added", "poor_households_added", "poor_people_added", "precision_poor", "high_need_households_added",
        "direct_support_coverage_before", "direct_support_coverage_after", "uncovered_high_need_before", "uncovered_high_need_after",
        "uncovered_high_need_reduction_pct", "additional_quarterly_transfer_rwf", "additional_annual_transfer_rwf"]
cov.loc[show]

In [ ]:
grid = pd.DataFrame([ps.coverage_expansion(df, x)["national"] for x in [0] + ps.COVERAGE_GRID])
fig, (a1, a2) = pl.figure("Coverage expansion: who the additional Direct Support would reach",
                          "Decision-support estimate. Left: high-need households still without any VUP component. Right: share of added households that are poor",
                          width=12, height=4.8, ncols=2, source="Source: EICV7 2023/24 (NISR), Umuseke out-of-sample model scores. Survey-weighted; static accounting, not causal impact.")
a1.plot(grid["increase_pp"], grid["uncovered_high_need_after"] / 1000, "-o", color=pl.SERIES[0], markersize=4, linewidth=2)
a1.set_xlabel("Direct Support coverage added, percentage points of all households", fontsize=9, color=pl.INK["secondary"])
a1.set_ylabel("Thousand households", fontsize=9, color=pl.INK["secondary"])
a2.plot(grid["increase_pp"].iloc[1:], 100 * grid["precision_poor"].iloc[1:], "-o", color=pl.SERIES[1], markersize=4, linewidth=2)
a2.set_ylim(0, 100)
a2.set_xlabel("Direct Support coverage added, percentage points", fontsize=9, color=pl.INK["secondary"])
a2.set_ylabel("% of added households that are poor", fontsize=9, color=pl.INK["secondary"])
for ax in (a1, a2):
    ax.grid(axis="both", color=pl.GRID)
path = pl.save(fig, FIG / "coverage_expansion.png", top=0.8, bottom=0.14, left=0.07, wspace=0.25)
display(Image(str(path)))
by = examples[2]["by_district_area"].groupby("district_name")[["households_added", "uncovered_high_need_before", "uncovered_high_need_after"]].sum()
by["reduction %"] = 100 * (1 - by["uncovered_high_need_after"] / by["uncovered_high_need_before"])
print("+2 pp: districts receiving the most new Direct Support households (weighted)")
by.sort_values("households_added", ascending=False).head(10)

## 2. Targeting: select the top 10%, 20% and 30% by poverty-model score

The scores are **out-of-sample**: cross-fitted for development households, and from the development-fitted model for holdout households. Precision and recall are therefore not inflated by training data.

In [ ]:
tg = {k: ps.targeting(df, k) for k in (0.10, 0.20, 0.30)}
nat = pd.DataFrame({f"top {int(100 * k)}%": r["national"] for k, r in tg.items()})
display(nat)
cols = ["sample_households", "selected_households", "poor_households_reached", "selection_rate", "precision", "recall_households", "recall_people", "share_of_selection"]
print("Top 20%: by urban/rural"); display(tg[0.20]["by_area"][cols])
print("Top 20%: by main income source"); display(tg[0.20]["by_income_source"][cols])
print("Top 20%: by district (sorted by poor households reached)")
tg[0.20]["by_district"][cols].sort_values("poor_households_reached", ascending=False)

In [ ]:
kk = np.linspace(0.05, 0.5, 10)
curve = pd.DataFrame([ps.targeting(df, k)["national"] for k in kk])
fig, (ax,) = pl.figure("Targeting budget: share of poor people reached and precision", "Decision-support estimate; out-of-sample poverty-model scores",
                       width=9, height=5, source="Source: EICV7 2023/24 (NISR), Umuseke model scores. Survey-weighted.")
ax.plot(100 * curve["k"], 100 * curve["recall_people"], "-o", color=pl.SERIES[0], linewidth=2, markersize=4, label="Poor people reached, %")
ax.plot(100 * curve["k"], 100 * curve["precision"], "-o", color=pl.SERIES[1], linewidth=2, markersize=4, label="Precision: selected households that are poor, %")
ax.axhline(100 * np.average(df["poor"], weights=w), linestyle=":", color=pl.BASELINE, label="Share of households that are poor (precision of random selection)")
ax.set_xlabel("Households selected, % of all households", fontsize=9, color=pl.INK["secondary"])
ax.set_ylim(0, 100); ax.grid(axis="both", color=pl.GRID); ax.legend(frameon=False, fontsize=8.5)
path = pl.save(fig, FIG / "targeting_budget.png", top=0.84, bottom=0.12, left=0.1)
display(Image(str(path)))

## 3. Benefit level: raise the NSDS quarterly payment

The additional value for **currently covered** NSDS households, using each household's observed quarterly payment. The annual value assumes four payments a year; a second estimate uses the quarters each household actually received, capped at four.

In [ ]:
ben = pd.DataFrame({f"+{x}%": ps.benefit_increase(df, x) for x in (10, 25, 50)})
ben

## 4. Protection gap: the 10 districts with the most uncovered high-need households

High-need households are the top quintile of the distress score excluding its VUP component. In each of the 10 districts with the most high-need households **without any VUP component**, the scenario newly covers *X*% of the district's high-need households, taken from the uncovered ones in priority order.

In [ ]:
gaps = ps.district_gaps(df)
display(gaps.head(12))
pg = {x: ps.protection_gap(df, x) for x in (25, 50)}
pgn = pd.DataFrame({f"{x}% of high-need": {k: v for k, v in r["national"].items() if k != "districts"} for x, r in pg.items()})
display(pgn)
t = pg[25]["by_district"]
t[t["targeted"]][["district_name", "high_need_households", "uncovered_high_need", "households_added", "uncovered_high_need_after", "protection_gap_rate", "protection_gap_rate_after"]]

In [ ]:
t = pg[25]["by_district"]
t = t[t["targeted"]].sort_values("uncovered_high_need")
fig, (ax,) = pl.figure("Protection gap in the 10 highest-gap districts: before and after the scenario",
                       "Uncovered high-need households (weighted, thousands). Scenario: newly cover 25% of each district's high-need households",
                       width=10, height=5.4, source="Source: EICV7 2023/24 (NISR). Decision-support estimate; survey-weighted; district estimates are imprecise.")
y = np.arange(len(t))
ax.barh(y + 0.18, t["uncovered_high_need"] / 1000, height=0.34, color=pl.NEUTRAL, label="Now")
ax.barh(y - 0.18, t["uncovered_high_need_after"] / 1000, height=0.34, color=pl.SERIES[0], label="After scenario")
ax.set_yticks(y, [f"{d} (n={n})" for d, n in zip(t["district_name"], t["high_need_sample"])], fontsize=9)
ax.set_xlabel("Thousand high-need households without any VUP component", fontsize=9, color=pl.INK["secondary"])
ax.legend(frameon=False, fontsize=9, loc="lower right")
path = pl.save(fig, FIG / "protection_gap.png", top=0.8, bottom=0.12, left=0.22)
display(Image(str(path)))

## 5. Uncertainty of the headline numbers

A cluster bootstrap (200 draws, clusters resampled within strata) reruns each scenario on resampled households. Ranking, budgets and allocation are recomputed in every draw. The intervals cover survey sampling only, not model or assumption error.

In [ ]:
rng = np.random.default_rng(2026)
idx = [[g.index.to_numpy() for _, g in d.groupby("clust")] for _, d in df.groupby("strata_id")]
draws = []
for b in range(200):
    rows = np.concatenate([np.concatenate([cl[i] for i in rng.integers(len(cl), size=len(cl))]) for cl in idx])
    d = df.iloc[rows].reset_index(drop=True)
    d["hhid"] = np.arange(len(d))
    c2, t20, g25 = ps.coverage_expansion(d, 2)["national"], ps.targeting(d, 0.20)["national"], ps.protection_gap(d, 25)["national"]
    draws.append({"coverage +2pp: poor households added": c2["poor_households_added"], "coverage +2pp: uncovered high-need after": c2["uncovered_high_need_after"],
                  "coverage +2pp: precision": c2["precision_poor"], "targeting 20%: precision": t20["precision"], "targeting 20%: poor people reached": t20["recall_people"],
                  "protection gap 25%: uncovered high-need after": g25["uncovered_high_need_after"], "protection gap 25%: households added": g25["households_added"]})
draws = pd.DataFrame(draws)
point = {"coverage +2pp: poor households added": examples[2]["national"]["poor_households_added"],
         "coverage +2pp: uncovered high-need after": examples[2]["national"]["uncovered_high_need_after"],
         "coverage +2pp: precision": examples[2]["national"]["precision_poor"], "targeting 20%: precision": tg[0.20]["national"]["precision"],
         "targeting 20%: poor people reached": tg[0.20]["national"]["recall_people"],
         "protection gap 25%: uncovered high-need after": pg[25]["national"]["uncovered_high_need_after"],
         "protection gap 25%: households added": pg[25]["national"]["households_added"]}
unc = pd.DataFrame({"estimate": point, "95% CI low": draws.quantile(0.025), "95% CI high": draws.quantile(0.975)})
unc.to_csv(ROOT / "reports" / "figures" / "policy_simulation" / "headline_uncertainty.csv")
unc

## 6. Save the scenario grid

In [ ]:
grid_all = ps.scenario_grid(df)
assert "hhid" not in grid_all.columns
grid_all.to_parquet(ps.SCENARIO_PATH, index=False)
print(f"{ps.SCENARIO_PATH.relative_to(ROOT)}: {len(grid_all):,} rows (aggregated only)")
display(grid_all.groupby(["scenario", "level"]).agg(rows=("value", "size"), parameter_values=("parameter_value", "nunique"), groups=("group", "nunique")))